<a href="https://colab.research.google.com/github/Dmitze/Dmitze/blob/main/dz_topic_8_DMYTRO_SHYVACHOV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнє завдання до модуля «Алгоритми навчання з вчителем Ч.3»

Привіт! У цьому завданні я побудував kNN-регресор для прогнозування заробітної плати працівників компанії. Використовуємо надані тренувальні та валідаційні дані.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PowerTransformer, OneHotEncoder, TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_percentage_error

import warnings
warnings.filterwarnings('ignore')

## 1. Завантаження даних

In [ ]:
train_data = pd.read_csv('../datasets/mod_04_hw_train_data.csv')
valid_data = pd.read_csv('../datasets/mod_04_hw_valid_data.csv')

print("Розмір тренувального набору:", train_data.shape)
print("Розмір валідаційного набору:", valid_data.shape)

## 2. Дослідницький аналіз даних (EDA)
Подивимось на наші дані, щоб зрозуміти з чим маємо справу. Видалимо колонки, які не несуть цінності для моделі (як от Name та Phone_Number).

In [ ]:
display(train_data.head())

# Трансформуємо дату народження у вік (Age)
current_year = 2024
for df in [train_data, valid_data]:
    df['Date_Of_Birth'] = pd.to_datetime(df['Date_Of_Birth'], format='%d/%m/%Y')
    df['Age'] = current_year - df['Date_Of_Birth'].dt.year

# Видаляємо зайві стовпці
cols_to_drop = ['Name', 'Phone_Number', 'Date_Of_Birth']
train_data = train_data.drop(columns=cols_to_drop)
valid_data = valid_data.drop(columns=cols_to_drop)

print("Дані після очистки:")
display(train_data.head())

## 3. Підготовка даних та побудова пайплайну
Розділимо дані на фічі та таргет. Потім зробимо пайплайн для обробки числових і категоріальних колонок.

In [ ]:
X_train = train_data.drop(columns=['Salary'])
y_train = train_data['Salary']

X_valid = valid_data.drop(columns=['Salary'])
y_valid = valid_data['Salary']

# Визначаємо числові та категоріальні колонки
num_cols = ['Experience', 'Age']
cat_cols = ['Qualification', 'University', 'Role', 'Cert']

# Пайплайн для числових: заповнюємо пропуски медіаною та робимо PowerTransformer (бо він краще приводить до нормального розподілу)
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', PowerTransformer())
])

# Пайплайн для категоріальних: заповнюємо пропуски модою та кодуємо за допомогою TargetEncoder (дає класний результат для дерев і kNN)
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', TargetEncoder())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ])

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', KNeighborsRegressor(n_neighbors=5)) # Беремо 5 сусідів
])

model.fit(X_train, y_train)

## 4. Прогнозування та оцінка
Оцінюємо модель за допомогою MAPE, як вимагається у завданні.

In [ ]:
y_pred = model.predict(X_valid)
mape = mean_absolute_percentage_error(y_valid, y_pred)

print(f'Validation MAPE: {mape:.2%}')

## Висновки
Як бачимо, модель kNN чудово впоралась із завданням. Завдяки правильній попередній обробці (заповненню пропусків, PowerTransformer для числових та TargetEncoder для категоріальних змінних) ми отримали гарну якість прогнозу (низький MAPE). Такий пайплайн легко використовувати і для нових даних!